# Per-Class Analysis + Statistical Significance

This notebook adds:
1. A single evaluation pass where both models are evaluated on the exact same, unshuffled test samples (so predictions are paired sample-for-sample) -- required for McNemar's test.
2. Per-class precision/recall/F1/support for each model, saved to a CSV.
3. Numeric confusion matrices plus the top confused class pairs per model.
4. **McNemar's test** on paired correct/incorrect outcomes (exact binomial for small discordant-pair counts, else continuity-corrected chi-square; implemented directly against `scipy.stats`, no `statsmodels` dependency).
5. A **paired bootstrap confidence interval** for the accuracy difference between the two models.

**Which checkpoints get compared** is controlled entirely by the `CNN_CKPT` / `MOBILENET_CKPT` cell below. Each checkpoint's actual architecture (`pooling`, `transformer_layers`) and training-time `motion_mode` are read back from the `config.json` saved next to it, so this works correctly for baseline checkpoints and for any ablation checkpoint (including a mismatched pair, e.g. CNN_LSTM trained with `optical_flow` compared against a MobileNet still on `diff`).

In [ ]:
import csv
import json
import os
import sys

import numpy as np
import torch
from scipy.stats import binom, chi2
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

BASE_DIR = os.path.abspath("..")
sys.path.append(BASE_DIR)

from models.cnn_lstm import CNN_LSTM
from models.mobile_net import MobileNetTransformer
from utils.dataloader import VideoDataset, get_transforms
from utils.metrics import compute_metrics

print("BASE_DIR:", BASE_DIR)

## Configuration -- pick which two checkpoints to compare

In [ ]:
def latest_run_checkpoint(log_dir):
    runs = [d for d in os.listdir(log_dir) if d.startswith("run_") and os.path.isdir(os.path.join(log_dir, d))]
    if not runs:
        raise FileNotFoundError(f"No run_* folders found in {log_dir}")
    latest = sorted(runs)[-1]
    return os.path.join(log_dir, latest, "best_model.pth")

# Edit these two paths to compare any pair of checkpoints.
# picks the most recently created run for each model -- ALWAYS double-check
# the printed paths below actually point at the pair you intend to compare.
CNN_CKPT = latest_run_checkpoint(os.path.join(BASE_DIR, "outputs/logs/cnn_lstm"))
MOBILENET_CKPT = latest_run_checkpoint(os.path.join(BASE_DIR, "outputs/logs/mobile_net"))

TEST_DIR = os.path.join(BASE_DIR, "data/WLBisindo/split/test")
BATCH_SIZE = 16
NUM_WORKERS = 0  # 0 is safest on Windows; raise if you know your setup handles it
NUM_BOOTSTRAP = 10000
SEED = 42

print("CNN_LSTM checkpoint   :", CNN_CKPT)
print("MobileNet checkpoint  :", MOBILENET_CKPT)

## Helper functions

In [ ]:
def load_run_config(ckpt_path):
    config_path = os.path.join(os.path.dirname(ckpt_path), "config.json")
    if not os.path.exists(config_path):
        return {}
    try:
        with open(config_path) as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def resolve_mobilenet_arch(ckpt_path):
    cfg = load_run_config(ckpt_path)
    pooling = cfg.get("pooling")
    if pooling not in ("last", "mean", "attention"):
        pooling = "last"
    num_layers = cfg.get("transformer_layers")
    if not isinstance(num_layers, int):
        num_layers = 4
    return pooling, num_layers


def resolve_motion_mode(ckpt_path, default="diff"):
    cfg = load_run_config(ckpt_path)
    motion_mode = cfg.get("motion_mode")
    if motion_mode not in ("diff", "optical_flow"):
        return default
    return motion_mode


def build_model(model_cls, ckpt_path, num_classes, device, **model_kwargs):
    model = model_cls(num_classes=num_classes, **model_kwargs)
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model


def exact_binomial_two_sided_pvalue(k, n, p=0.5):
    if n == 0:
        return 1.0
    cdf_k = binom.cdf(k, n, p)
    sf_k_minus_1 = 1.0 - binom.cdf(k - 1, n, p) if k > 0 else 1.0
    return float(min(1.0, 2.0 * min(cdf_k, sf_k_minus_1)))


def mcnemar_test(correct_a, correct_b, model_a_name, model_b_name):
    """b = A correct & B wrong ; c = A wrong & B correct (discordant pairs).
    Uses exact binomial when b + c < 25, else continuity-corrected chi-square."""
    correct_a = np.asarray(correct_a, dtype=bool)
    correct_b = np.asarray(correct_b, dtype=bool)

    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    both_right = int(np.sum(correct_a & correct_b))
    both_wrong = int(np.sum(~correct_a & ~correct_b))
    n_discordant = b + c

    if n_discordant == 0:
        method, statistic, p_value = "undefined (no discordant pairs)", 0.0, 1.0
    elif n_discordant < 25:
        method = "exact_binomial"
        statistic = float(b)
        p_value = exact_binomial_two_sided_pvalue(min(b, c), n_discordant, p=0.5)
    else:
        method = "chi_square_continuity_corrected"
        statistic = float((abs(b - c) - 1) ** 2 / n_discordant)
        p_value = float(1.0 - chi2.cdf(statistic, df=1))

    if b == c:
        favors = "tie (equal discordant wins)"
    elif b > c:
        favors = f"{model_a_name} wins more discordant pairs ({b} vs {c})"
    else:
        favors = f"{model_b_name} wins more discordant pairs ({c} vs {b})"

    return {
        "contingency_table": {
            "both_correct": both_right,
            f"{model_a_name}_correct_only": b,
            f"{model_b_name}_correct_only": c,
            "both_wrong": both_wrong,
        },
        "n_discordant": n_discordant,
        "method": method,
        "statistic": statistic,
        "p_value": p_value,
        "significant_at_0.05": bool(p_value < 0.05),
        "favors": favors,
    }


def paired_bootstrap_accuracy_diff(correct_a, correct_b, num_bootstrap, seed):
    rng = np.random.default_rng(seed)
    correct_a = np.asarray(correct_a, dtype=bool)
    correct_b = np.asarray(correct_b, dtype=bool)
    n = len(correct_a)

    diffs = np.empty(num_bootstrap, dtype=np.float64)
    for i in range(num_bootstrap):
        idx = rng.integers(0, n, size=n)
        diffs[i] = correct_a[idx].mean() - correct_b[idx].mean()

    observed_diff = float(correct_a.mean() - correct_b.mean())
    lower, upper = np.percentile(diffs, [2.5, 97.5])
    p_value = float(min(1.0, 2.0 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))))

    return {
        "observed_accuracy_diff": observed_diff,
        "bootstrap_mean_diff": float(diffs.mean()),
        "ci_95_lower": float(lower),
        "ci_95_upper": float(upper),
        "p_value_approx": p_value,
        "significant_at_0.05": bool(not (lower <= 0 <= upper)),
        "num_bootstrap": num_bootstrap,
    }


def top_confused_pairs(cm, class_names, top_k=10):
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                pairs.append({"true_class": class_names[i], "predicted_as": class_names[j], "count": int(cm[i, j])})
    pairs.sort(key=lambda d: d["count"], reverse=True)
    return pairs[:top_k]

## Build test datasets and load models

Built per-model (not shared) because the two checkpoints may have been trained with different `motion_mode`. If they match, the same dataset object is reused. Either way, sample ordering is verified to be identical before any paired statistic is computed.

In [ ]:
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mobilenet_pooling, mobilenet_num_layers = resolve_mobilenet_arch(MOBILENET_CKPT)
cnn_motion_mode = resolve_motion_mode(CNN_CKPT)
mobilenet_motion_mode = resolve_motion_mode(MOBILENET_CKPT)
print("CNN_LSTM motion_mode        :", cnn_motion_mode)
print("MobileNet motion_mode       :", mobilenet_motion_mode)
print("MobileNet pooling/layers    :", mobilenet_pooling, "/", mobilenet_num_layers)

_, val_transform = get_transforms()

cnn_test_dataset = VideoDataset(TEST_DIR, transform=val_transform, motion_mode=cnn_motion_mode)
cnn_test_loader = DataLoader(cnn_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

if mobilenet_motion_mode == cnn_motion_mode:
    mobilenet_test_dataset = cnn_test_dataset
    mobilenet_test_loader = cnn_test_loader
else:
    mobilenet_test_dataset = VideoDataset(TEST_DIR, transform=val_transform, motion_mode=mobilenet_motion_mode)
    mobilenet_test_loader = DataLoader(mobilenet_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

assert cnn_test_dataset.samples == mobilenet_test_dataset.samples, (
    "CNN_LSTM and MobileNetTransformer test datasets do not enumerate samples in the same "
    "order -- aborting rather than silently mis-pairing predictions."
)

idx_to_name = {idx: name for name, idx in cnn_test_dataset.label_map.items()}
num_classes = len(cnn_test_dataset.label_map)
class_names = [idx_to_name[i] for i in range(num_classes)]
print("Num classes  :", num_classes)
print("Test samples :", len(cnn_test_dataset))

cnn_model = build_model(CNN_LSTM, CNN_CKPT, num_classes, device)
mobilenet_model = build_model(MobileNetTransformer, MOBILENET_CKPT, num_classes, device, pooling=mobilenet_pooling, num_layers=mobilenet_num_layers)

## Run inference on the test set

In [ ]:
y_true_cnn, pred_cnn = [], []
with torch.no_grad():
    for x, y in cnn_test_loader:
        x = x.to(device, non_blocking=True)
        y_true_cnn.extend(y.numpy().tolist())
        out_cnn = cnn_model(x)
        pred_cnn.extend(torch.argmax(out_cnn, dim=1).cpu().numpy().tolist())

y_true_mobilenet, pred_mobilenet = [], []
with torch.no_grad():
    for x, y in mobilenet_test_loader:
        x = x.to(device, non_blocking=True)
        y_true_mobilenet.extend(y.numpy().tolist())
        out_mobilenet = mobilenet_model(x)
        pred_mobilenet.extend(torch.argmax(out_mobilenet, dim=1).cpu().numpy().tolist())

y_true_cnn = np.array(y_true_cnn)
y_true_mobilenet = np.array(y_true_mobilenet)
assert np.array_equal(y_true_cnn, y_true_mobilenet), (
    "Ground-truth label sequences differ between the two evaluation passes -- pairing "
    "assumption violated."
)
y_true = y_true_cnn
pred_cnn = np.array(pred_cnn)
pred_mobilenet = np.array(pred_mobilenet)

print("Inference done. Samples:", len(y_true))

## Overall metrics (sanity check against test_eval.json)

In [ ]:
metrics_cnn = compute_metrics(y_true, pred_cnn)
metrics_mobilenet = compute_metrics(y_true, pred_mobilenet)
print("CNN_LSTM            :", metrics_cnn)
print("MobileNetTransformer:", metrics_mobilenet)

## Per-class report

In [ ]:
report_cnn = classification_report(y_true, pred_cnn, labels=list(range(num_classes)), target_names=class_names, output_dict=True, zero_division=0)
report_mobilenet = classification_report(y_true, pred_mobilenet, labels=list(range(num_classes)), target_names=class_names, output_dict=True, zero_division=0)

per_class_rows = []
for name in class_names:
    cnn_row = report_cnn[name]
    mob_row = report_mobilenet[name]
    per_class_rows.append({
        "class_name": name,
        "support": int(cnn_row["support"]),
        "cnn_precision": cnn_row["precision"],
        "cnn_recall": cnn_row["recall"],
        "cnn_f1": cnn_row["f1-score"],
        "mobilenet_precision": mob_row["precision"],
        "mobilenet_recall": mob_row["recall"],
        "mobilenet_f1": mob_row["f1-score"],
        "f1_diff_mobilenet_minus_cnn": mob_row["f1-score"] - cnn_row["f1-score"],
    })
per_class_rows.sort(key=lambda r: r["f1_diff_mobilenet_minus_cnn"])

import pandas as pd
pd.DataFrame(per_class_rows)

## Confusion matrices + top confused pairs

In [ ]:
cm_cnn = confusion_matrix(y_true, pred_cnn, labels=list(range(num_classes)))
cm_mobilenet = confusion_matrix(y_true, pred_mobilenet, labels=list(range(num_classes)))
confused_cnn = top_confused_pairs(cm_cnn, class_names, top_k=10)
confused_mobilenet = top_confused_pairs(cm_mobilenet, class_names, top_k=10)

print("Top confused class pairs (CNN_LSTM):")
for p in confused_cnn[:5]:
    print(f"  {p['true_class']} -> predicted as {p['predicted_as']} ({p['count']}x)")
print("Top confused class pairs (MobileNetTransformer):")
for p in confused_mobilenet[:5]:
    print(f"  {p['true_class']} -> predicted as {p['predicted_as']} ({p['count']}x)")

## McNemar's test

In [ ]:
correct_cnn = (pred_cnn == y_true)
correct_mobilenet = (pred_mobilenet == y_true)
mcnemar_result = mcnemar_test(correct_cnn, correct_mobilenet, "CNN_LSTM", "MobileNetTransformer")
print(json.dumps(mcnemar_result, indent=2))

## Paired bootstrap CI on accuracy difference

In [ ]:
bootstrap_result = paired_bootstrap_accuracy_diff(correct_cnn, correct_mobilenet, NUM_BOOTSTRAP, SEED)
print(json.dumps(bootstrap_result, indent=2))

## Save results

Saved with a filename derived from both checkpoints' run names, so comparing multiple pairs doesn't overwrite previous results.

In [ ]:
cnn_run_name = os.path.basename(os.path.dirname(CNN_CKPT))
mobilenet_run_name = os.path.basename(os.path.dirname(MOBILENET_CKPT))

results = {
    "test_dir": TEST_DIR,
    "cnn_ckpt": CNN_CKPT,
    "mobilenet_ckpt": MOBILENET_CKPT,
    "cnn_motion_mode": cnn_motion_mode,
    "mobilenet_motion_mode": mobilenet_motion_mode,
    "mobilenet_pooling": mobilenet_pooling,
    "mobilenet_num_layers": mobilenet_num_layers,
    "num_test_samples": int(len(cnn_test_dataset)),
    "num_classes": num_classes,
    "overall_metrics": {"CNN_LSTM": metrics_cnn, "MobileNetTransformer": metrics_mobilenet},
    "mcnemar_test": mcnemar_result,
    "paired_bootstrap_accuracy_diff": bootstrap_result,
    "confusion_matrix": {"class_order": class_names, "CNN_LSTM": cm_cnn.tolist(), "MobileNetTransformer": cm_mobilenet.tolist()},
    "top_confused_pairs": {"CNN_LSTM": confused_cnn, "MobileNetTransformer": confused_mobilenet},
    "methodology_note": (
        "Both models were evaluated on identical (or index-verified equivalent) unshuffled "
        "test batches, so predictions are paired sample-for-sample. McNemar's test uses the "
        "exact binomial form when discordant pairs < 25, else continuity-corrected chi-square "
        "(no statsmodels dependency). The bootstrap CI resamples test indices jointly for both "
        "models to preserve pairing."
    ),
}

out_dir = os.path.join(BASE_DIR, "outputs/metrics")
os.makedirs(out_dir, exist_ok=True)

json_path = os.path.join(out_dir, f"statistical_analysis__{cnn_run_name}__vs__{mobilenet_run_name}.json")
with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

csv_path = os.path.join(out_dir, f"per_class_report__{cnn_run_name}__vs__{mobilenet_run_name}.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(per_class_rows[0].keys()))
    writer.writeheader()
    writer.writerows(per_class_rows)

print("Saved detailed results to:", json_path)
print("Saved per-class report to:", csv_path)